# 📋 Odpowiedzi: Regresja logistyczna

*Wygenerowano automatycznie — nie commitować!*

---
### 1. Ćwiczenie 1

In [ ]:
#rozwiązanie
random.seed(42)
random.shuffle(all_data_processed)

---
### 2. Sprawdźmy rozmiary naszych X'ów i Y'ków

In [ ]:
# rozwiązanie
X_train = X[:split_idx,:,:,:]
X_test = X[split_idx:,:,:,:]
Y_train = Y[:split_idx]
Y_test = Y[split_idx:]

---
### 3. Proszę uzupełnić definicję funkcji sigmoid:

Funkcja sigmoidalna przekształca dowolną liczbę rzeczywistą w wartość z zakresu $(0, 1)$, co pozwala interpretować wynik jako prawdopodobieństwo:

$$\sigma(z) = \frac{1}{1 + e^{-z}}$$

gdzie $z = w^T x + b$.

In [ ]:
# rozwiązanie
def sigmoid(z):

    s = 1 / (1 + np.exp(-z))

    return s

---
### 4. Proszę uzupełnić funkcję inicjalizacji parametrów

Chodzi po prostu o to aby zainicjalizowane macierze *w* i *b* były odpowiednich rozmiarów. Dla regresji logistycznej zainicjalizujemy wszystko na wartość 0, **inaczej niż w sieciach neuronowych gdzie będziemy inicjalizować za pomocą liczby losowej**.

In [ ]:
# rozwiązanie
def initialize_with_zeros(dim):

    w = np.zeros(shape=(dim, 1))
    b = 0

    assert(w.shape == (dim, 1))
    assert(isinstance(b, float) or isinstance(b, int))
    
    return w, b

---
### 5. Proszę uzupełnić kod zawierający Forward propagation i Backward propagation

Czyli wyliczenie kosztu podczas propagacji wprzód i wyliczenie gradientu podczas propagacji wstecz. Do poniższej funkcji przydadzą się następujące informacje:

Forward Propagation:
- Na wejściu mamy macierz X
- Wyliczamy aktualną predykcję modelu (dla wszystkich $m$ przykładów) $A = \sigma(w^T X + b) = (a^{(0)}, a^{(1)}, ..., a^{(m-1)}, a^{(m)})$
- Wyliczamy aktualną wartość funkcji kosztu: $J = -\frac{1}{m}\sum_{i=1}^{m}y^{(i)}\log(a^{(i)})+(1-y^{(i)})\log(1-a^{(i)})$

Backward Propagation - na podstawie aktualnej funkcji kosztu wyliczamy gradient (pochodne cząstkowe) - te gradienty wskarzą nam w jaką stronę mamy podążać aby zmniejszać funkcję kosztu: 

$$ \frac{\partial J}{\partial w} = \frac{1}{m}X(A-Y)^T$$
$$ \frac{\partial J}{\partial b} = \frac{1}{m} \sum_{i=1}^m (a^{(i)}-y^{(i)})$$

In [ ]:
# rozwiązanie
def propagate(w, b, X, Y):

    m = X.shape[1]

    A = sigmoid(np.dot(w.T, X) + b)  # wyliczyć aktywację
    cost = (- 1 / m) * np.sum(Y * np.log(A) + (1 - Y) * (np.log(1 - A)))  # wyliczyć funkcję kosztu

    dw = (1 / m) * np.dot(X, (A - Y).T)
    db = (1 / m) * np.sum(A - Y)

    assert(dw.shape == w.shape)
    assert(db.dtype == float)
    cost = np.squeeze(cost)
    assert(cost.shape == ())
    
    grads = {"dw": dw,
             "db": db}
    
    return grads, cost

---
### 6. Optymalizacja
Przypomnijmy sobie co już mamy gotowe
- Zainicjowaliśmy parametry.
- Posiadamy już funkcję do wyliczania kosztu i gradientu.
- Zatem teraz na tej podstawie będziemy aktualizować parametry.

Do powyższej zdefiniowanego zadania potrzebujemy jeszcze znać liczbę powtórzeń - num_iterations, nazywana też epokami w ML oraz kroku uczenia (learning rate) - czyli jak bardzo będziemy się poruszali w stronę wyznaczonego gradientu.

Wzory do aktulizacji parametrów: $$ w = w - \alpha \text{ } dw$$ $$ b = b - \alpha \text{ } db$$ gdzie $\alpha$ jest krokiem uczenia.

In [ ]:
#rozwiązanie
def optimize(w, b, X, Y, num_iterations, learning_rate, print_cost = False):
    
    costs = []
    
    for i in range(num_iterations):
        
        grads, cost = propagate(w, b, X, Y)
        
        dw = grads["dw"]
        db = grads["db"]

        w = w - learning_rate * dw  
        b = b - learning_rate * db
        
        if i % 100 == 0:
            costs.append(cost)
        
        if print_cost and i % 100 == 0:
            print (f"Koszt po {i} iteracji: {cost}")
    
    params = {"w": w,
              "b": b}
    
    grads = {"dw": dw,
             "db": db}
    
    return params, grads, costs

---
### 7. Sprawdźmy

In [ ]:
# rozwiązanie
def predict(w, b, X):
   
    m = X.shape[1]
    Y_prediction = np.zeros((1, m))
    w = w.reshape(X.shape[0], 1)
    
    A = sigmoid(np.dot(w.T, X) + b)
    
    for i in range(A.shape[1]):
        Y_prediction[0, i] = 1 if A[0, i] > 0.5 else 0
    
    assert(Y_prediction.shape == (1, m))
    
    return Y_prediction

---
### 8. Model
Teraz musimy połączyć napisane wcześniej elementy (funkcje) w cały działający model. Większość uzupełniania poniżej polega na odpowiednim wywołaniu wcześniej napisanych funkcji.
<br>Aby zbiorczo mierzyć (i porównywać między sobą) modele w ML stosujemy wiele różnych miar. Jedną z najbardziej popularnych jest dokładność (accuracy). Jest to ułamek/część z całości, która została poprawnie zaklasyfikowana przez model:
$$ Dokładność \; (Accuracy) = \frac{Liczba \;poprawnych \;predykcji}{Liczba \;wszystkich \;predykcji}$$

In [ ]:
# rozwiązanie
def model(X_train, Y_train, X_test, Y_test, num_iterations=2000, learning_rate=0.5, print_cost=False):

    w, b = initialize_with_zeros(X_train.shape[0])

    parameters, grads, costs = optimize(w, b, X_train, Y_train, num_iterations, learning_rate, print_cost)
    
    w = parameters["w"]
    b = parameters["b"]
    
    Y_prediction_test = predict(w, b, X_test)
    Y_prediction_train = predict(w, b, X_train)

    print(f"Precyzja na zbiorze treningowym (train accuracy): {100 - np.mean(np.abs(Y_prediction_train - Y_train)) * 100}")
    print(f"Precyzja na zbiorze testowym (test accuracy): {100 - np.mean(np.abs(Y_prediction_test - Y_test)) * 100}")

    
    d = {"costs": costs,
         "Y_prediction_test": Y_prediction_test, 
         "Y_prediction_train" : Y_prediction_train, 
         "w" : w, 
         "b" : b,
         "learning_rate" : learning_rate,
         "num_iterations": num_iterations}
    
    return d